In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import os, getpass, subprocess
from google.colab import files

USER = "sristipokharel9"
REPO = "placement-analytics"

# The token is typed into a hidden box at run time and is never saved in the notebook
TOKEN = getpass.getpass("Paste GitHub token: ")

!git clone https://{TOKEN}@github.com/{USER}/{REPO}.git
%cd {REPO}
!git config user.name "Your Name"
!git config user.email "you@example.com"

for d in ["data", "images", "models"]:
    os.makedirs(d, exist_ok=True)

def push(msg):
    for cmd in (["git","add","."], ["git","commit","-m",msg], ["git","push"]):
        r = subprocess.run(cmd, capture_output=True, text=True)
        print(r.stdout, r.stderr)

In [ ]:
up = files.upload()
for f in up: os.replace(f, f"data/{f}")
df = pd.read_csv("data/DS_Day01_48_Placement_Analytics_Dataset.csv")
df.head()

In [ ]:
print(df.shape)

In [ ]:
push("Add raw dataset")

In [ ]:
print("Shape:", df.shape)
print("\nMissing values:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate Student_IDs:", df.Student_ID.duplicated().sum())
df.info()

In [ ]:
num_cols = ["Academic_Performance","Coding_Score","Project_Score",
            "Certifications","Internships","Communication_Score"]

# fill with the median (robust to skew)
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# counts must be whole numbers
df[["Certifications","Internships"]] = df[["Certifications","Internships"]].round()

print("Missing after cleaning:", df.isna().sum().sum())

In [ ]:
score_cols = ["Academic_Performance","Coding_Score","Project_Score","Communication_Score"]

print("Scores outside 0-100:", ((df[score_cols] < 0) | (df[score_cols] > 100)).sum().sum())
print("Negative certs/internships:", ((df[["Certifications","Internships"]] < 0).sum().sum()))
print("Placed but 0 offers:", ((df.Placement_Status=="Placed") & (df.Offer_Count==0)).sum())
print("Not placed but offers/salary > 0:",
      ((df.Placement_Status=="Not Placed") & ((df.Offer_Count>0) | (df.Salary>0))).sum())
print("Placed but salary = 0:", ((df.Placement_Status=="Placed") & (df.Salary==0)).sum())

In [ ]:
df["Placed"] = (df.Placement_Status == "Placed").astype(int)
df.to_csv("data/placement_cleaned.csv", index=False)
print(df.Placed.value_counts())

In [ ]:
push("Clean and validate dataset")

In [ ]:
display(df.describe().T.round(2))
print("Overall placement rate:", round(df.Placed.mean()*100, 1), "%")
print("\nAverage salary of placed students:", round(df[df.Placed==1].Salary.mean(), 2))
print("\nOffer count distribution:")
print(df.Offer_Count.value_counts().sort_index())

In [ ]:
display(df.groupby("Placement_Status")[num_cols].mean().round(2))

display(df.groupby("Department").agg(
    Students=("Placed","count"),
    Placement_Rate=("Placed","mean"),
    Avg_Offers=("Offer_Count","mean")).round(2).sort_values("Placement_Rate"))

In [ ]:
from scipy import stats

corr = df[num_cols + ["Placed"]].corr()["Placed"].drop("Placed").sort_values(ascending=False)
print("Correlation with Placed:\n", corr.round(3), "\n")

for c in num_cols:
    a = df[df.Placed==1][c]; b = df[df.Placed==0][c]
    t, p = stats.ttest_ind(a, b, equal_var=False)
    print(f"{c:22s} placed={a.mean():6.2f}  not placed={b.mean():6.2f}  p-value={p:.4g}")

In [ ]:
push("EDA: descriptive stats and placement factors")

In [ ]:
df.groupby("Placement_Status")[num_cols].mean().round(2).to_csv("data/eda_placed_vs_not.csv")
df.groupby("Department").agg(Students=("Placed","count"),
    Placement_Rate=("Placed","mean"), Avg_Offers=("Offer_Count","mean")).round(2).to_csv("data/eda_by_department.csv")
push("EDA: descriptive stats and placement factors")

In [ ]:
df["Offer_Group"] = pd.cut(df.Offer_Count, [-1,0,1,10], labels=["None","Single","Multiple"])

print(df.Offer_Group.value_counts())
display(df.groupby("Offer_Group", observed=True)[num_cols + ["Salary"]].mean().round(2))

In [ ]:
placed = df[df.Placed==1].copy()

print("Pearson :", round(placed.Coding_Score.corr(placed.Salary), 3))
print("Spearman:", round(placed.Coding_Score.corr(placed.Salary, method="spearman"), 3))

placed["Coding_Band"] = pd.cut(placed.Coding_Score, [0,60,70,80,101],
                               labels=["<60","60-70","70-80","80+"])
display(placed.groupby("Coding_Band", observed=True).Salary.agg(["count","mean"]).round(2))

In [ ]:
plt.figure(figsize=(7,5))
sns.regplot(data=placed, x="Coding_Score", y="Salary",
            scatter_kws={"alpha":0.3}, line_kws={"color":"red"})
plt.title("Coding Score vs Salary (Placed Students)")
plt.xlabel("Coding Score"); plt.ylabel("Salary (LPA)")
plt.savefig("images/1_coding_vs_salary.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
push("Multiple-offer analysis and coding vs salary (viz 1)")

In [ ]:
df["Acad_Band"] = pd.qcut(df.Academic_Performance, 3, labels=["Low","Medium","High"])
df["Cert_Group"] = pd.cut(df.Certifications, [-1,0,2,20], labels=["0","1-2","3+"])

print(df.Acad_Band.value_counts())
print(df.Cert_Group.value_counts())

In [ ]:
print("Placement rate (%)")
display((df.pivot_table(index="Acad_Band", columns="Cert_Group",
         values="Placed", aggfunc="mean", observed=True) * 100).round(1))

In [ ]:
print("Average salary (placed students only)")
display(df[df.Placed==1].pivot_table(index="Acad_Band", columns="Cert_Group",
        values="Salary", aggfunc="mean", observed=True).round(2))

In [ ]:
import statsmodels.formula.api as smf

m = smf.ols("Salary ~ Academic_Performance + Certifications",
            data=df[df.Placed==1]).fit()
print(m.summary().tables[1])

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df[df.Placed==1], x="Cert_Group", y="Salary", hue="Acad_Band")
plt.title("Salary by Certifications within Academic Bands")
plt.xlabel("Number of Certifications"); plt.ylabel("Salary (LPA)")
plt.savefig("images/2_certifications_value.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
push("Certification value analysis (viz 2)")

In [ ]:
dept = df.groupby("Department").agg(
    Students=("Placed","count"),
    Placement_Rate=("Placed","mean"),
    Avg_Offers=("Offer_Count","mean"),
    Avg_Salary_Placed=("Salary", lambda s: s[s>0].mean()),
    Academic=("Academic_Performance","mean"),
    Coding=("Coding_Score","mean"),
    Project=("Project_Score","mean"),
    Communication=("Communication_Score","mean"),
    Certifications=("Certifications","mean"),
    Internships=("Internships","mean")).round(2)

display(dept.sort_values("Placement_Rate"))
dept.to_csv("data/department_summary.csv")

In [ ]:
skills = ["Academic","Coding","Project","Communication","Certifications","Internships"]
gap = (dept[skills] - dept[skills].mean()).round(2)
display(gap)


In [ ]:
plt.figure(figsize=(8,5))
(dept.Placement_Rate*100).sort_values().plot.barh(color="steelblue")
plt.xlabel("Placement Rate (%)")
plt.title("Placement Rate by Department")
plt.savefig("images/3_dept_placement_rate.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(9,5))
sns.heatmap(dept[["Coding","Project","Communication","Certifications","Internships"]],
            annot=True, cmap="YlGnBu", fmt=".1f")
plt.title("Skill Profile by Department")
plt.savefig("images/4_dept_skill_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df["Profile"] = "Balanced"
df.loc[(df.Coding_Score>=80) & (df.Communication_Score<70), "Profile"] = "Coder, weak communicator"
df.loc[(df.Coding_Score<65) & (df.Communication_Score>=80), "Profile"] = "Communicator, weak coder"
df.loc[(df.Coding_Score<65) & (df.Communication_Score<70), "Profile"] = "Weak on both"
df.loc[(df.Coding_Score>=80) & (df.Communication_Score>=80), "Profile"] = "Strong on both"

display(df.groupby("Profile").agg(
    Students=("Placed","count"),
    Placement_Rate=("Placed","mean"),
    Avg_Offers=("Offer_Count","mean")).round(2))

In [ ]:
push("Department comparison and skill profiles (viz 3 and 4)")

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score

X = pd.get_dummies(df[num_cols + ["Department"]], columns=["Department"], drop_first=True)
y = df["Placed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Placed share in train:", round(y_train.mean(), 3), " test:", round(y_test.mean(), 3))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
print("Model trained")

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

pred  = rf.predict(X_test)
proba = rf.predict_proba(X_test)[:, 1]

report = classification_report(y_test, pred, target_names=["Not Placed","Placed"])
auc = roc_auc_score(y_test, proba)

print(report)
print("ROC-AUC:", round(auc, 3))

In [ ]:
cv_f1  = cross_val_score(rf, X, y, cv=5, scoring="f1").mean()
cv_auc = cross_val_score(rf, X, y, cv=5, scoring="roc_auc").mean()
print("5-fold CV F1 :", round(cv_f1, 3))
print("5-fold CV AUC:", round(cv_auc, 3))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(rf, X_test, y_test,
    display_labels=["Not Placed","Placed"], cmap="Blues")
plt.title("Confusion Matrix (Test Set)")
plt.savefig("images/5_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
imp = pd.Series(rf.feature_importances_, X.columns).sort_values()
imp.plot.barh(figsize=(7,6), title="Random Forest Feature Importance")
plt.xlabel("Importance")
plt.savefig("images/6_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print(imp.sort_values(ascending=False).round(3))

In [ ]:
import joblib
joblib.dump(rf, "models/random_forest.pkl")

with open("models/metrics.txt", "w") as f:
    f.write(report)
    f.write(f"\nROC-AUC (test): {auc:.3f}")
    f.write(f"\n5-fold CV F1: {cv_f1:.3f}")
    f.write(f"\n5-fold CV AUC: {cv_auc:.3f}\n")
    f.write("\nFeature importance:\n" + imp.sort_values(ascending=False).round(3).to_string())

In [ ]:
push("Train and evaluate Random Forest model")

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

rows = []
for t in [0.5, 0.6, 0.7, 0.75, 0.8, 0.85]:
    pred_t = (proba >= t).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y_test, pred_t, labels=[0], zero_division=0)
    rows.append({"threshold": t, "NotPlaced_precision": p[0].round(2),
                 "NotPlaced_recall": r[0].round(2), "NotPlaced_f1": f[0].round(2)})
display(pd.DataFrame(rows))

In [ ]:
%%writefile README.md
# Placement Analytics: What Skills Lead to Better Offers?

## Dataset
1,500 students across 7 departments. Missing values (8 per numeric column) were filled with the median. No duplicates or invalid values were found.

## Key Insights
1. Overall placement rate is 82.8% (1,242 of 1,500).
2. Coding, project and academic scores are the top drivers of placement.
3. Multiple-offer students have higher coding scores (76.2 vs 66.6 for unplaced) and more internships.
4. Coding strongly correlates with salary (r = 0.64); salary rises from 3.81 LPA (<60) to 6.53 LPA (80+).
5. Certifications add value beyond marks: about +0.21 LPA each, and placement rises from 67.6% to 87.8% for low-marks students with 3+ certs.
6. Departments differ little (81% to 86% placement), so skill gaps matter more than department.
7. Communication scores are uniformly high and less differentiating.

## Model
Random Forest Classifier (300 trees, balanced class weights). Target: Placement_Status.
Offer_Count and Salary were excluded to avoid data leakage.
Test ROC-AUC about 0.72. At the default 0.5 cut-off, recall for Not Placed was only 0.02, so the
decision threshold was lowered to [YOUR CHOICE] to catch more at-risk students (recall [X], precision [Y]).
Details: models/metrics.txt

## Action Plan
- Coding practice programme for students below 70
- Certification drive (3+ per student)
- Mandatory projects and at least one internship
- Model-based early-warning list of at-risk students
- College-wide skill programmes rather than department-specific ones

## Visualisations
See the images/ folder.

In [ ]:
push("Add threshold tuning, README with insights and action plan")

In [ ]:
THRESH = 0.8
pred_final = (proba >= THRESH).astype(int)
print(classification_report(y_test, pred_final, target_names=["Not Placed","Placed"]))

with open("models/metrics.txt", "a") as f:
    f.write(f"\n\nTuned threshold = {THRESH}\n")
    f.write(classification_report(y_test, pred_final, target_names=["Not Placed","Placed"]))

In [ ]:
%%writefile README.md
# Placement Analytics: What Skills Lead to Better Offers?

## Dataset
1,500 students across 7 departments. Missing values (8 per numeric column) were filled with the median. No duplicates or invalid values were found.

## Key Insights
1. Overall placement rate is 82.8% (1,242 of 1,500).
2. Coding, project and academic scores are the top drivers of placement.
3. Multiple-offer students have higher coding scores (76.2 vs 66.6 for unplaced) and more internships.
4. Coding strongly correlates with salary (r = 0.64); salary rises from 3.81 LPA (<60) to 6.53 LPA (80+).
5. Certifications add value beyond marks: about +0.21 LPA each, and placement rises from 67.6% to 87.8% for low-marks students with 3+ certs.
6. Departments differ little (81% to 86% placement), so skill gaps matter more than department.
7. Communication scores are uniformly high and less differentiating.

## Model
Random Forest Classifier (300 trees, balanced class weights). Target: Placement_Status.
Offer_Count and Salary were excluded to avoid data leakage.
Test ROC-AUC is about 0.72, so predictive power is moderate. At the default 0.5 cut-off, recall for
Not Placed was only 0.02, so the decision threshold was lowered to 0.8. At 0.8 the model catches
about 65% of at-risk students (precision about 0.33), which suits an early-warning support list
but means roughly two of three flagged students would have been placed anyway.
Details: models/metrics.txt

## Action Plan
- Coding practice programme for students below 70
- Certification drive (3+ per student)
- Mandatory projects and at least one internship
- Model-based early-warning list of at-risk students
- College-wide skill programmes rather than department-specific ones

## Visualisations
See the images/ folder.

In [ ]:
push("Add tuned threshold, README with insights and action plan")